# Notebook 08 — Results Visualization

## Purpose

Draw the figures used in the report from tables that earlier notebooks already saved. This notebook does not fit a new model and does not change a selected result.

Each figure states the denominator or the score it plots. Topic-model coherence, held-out macro-F1, and refit agreement are different quantities and are drawn on separate figures.


## 1. Read the saved result tables

Draw figures only from tables that earlier notebooks have already written.


In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Project root not found.")

TOPIC = ROOT / "outputs" / "04_topic_modeling" / "20261019_topic_models"
MODEL = ROOT / "outputs" / "05_model_development" / "20261019_supervised"
PRED = ROOT / "outputs" / "06_results_predictions" / "20261019_predictions"
EVENTS = ROOT / "outputs" / "07_events_timeline" / "20261019_events"
RUN_ID = "20261019_figures"
OUT = ROOT / "outputs" / "08_results_visualization" / RUN_ID
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

comparison = pd.read_csv(TOPIC / "topic_model_comparison.csv")
topic_terms = pd.read_csv(TOPIC / "selected_topic_terms.csv")
validation = pd.read_csv(MODEL / "validation_scores.csv")
heldout = pd.read_csv(MODEL / "heldout_test_scores.csv")
agreement = pd.read_csv(PRED / "refit_agreement.csv")
distribution = pd.read_csv(PRED / "label_distribution_comparison.csv")
timeline = pd.read_csv(EVENTS / "post_timeline.csv")
print("Figure directory:", FIG)


Figure directory: D:\NIRA_CASE_STUDY\outputs\08_results_visualization\20261019_figures\figures


### Interpretation

The figure notebook reads the topic comparison, the validation scores, the held-out scores, the prediction tables, and the post timeline. It does not fit a new model or change a selected result.


## 2. Draw the six report figures

Keep topic coherence, held-out macro-F1, and refit agreement on separate figures.


In [2]:
plot_df = comparison.dropna(subset=["coherence_c_npmi"]).copy()
plot_df["label"] = plot_df["family"] + " " + plot_df["setting"].astype(str)
plot_df = plot_df.sort_values("coherence_c_npmi")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["label"], plot_df["coherence_c_npmi"], color="#1f4e79")
ax.set_xlabel("c_npmi coherence")
ax.set_title("Topic models on all 5,292 comments")
fig.tight_layout()
fig.savefig(FIG / "topic_coherence.png", dpi=150)
plt.close(fig)

sizes = topic_terms.sort_values("n_comments")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(sizes["topic_id"].astype(str), sizes["n_comments"], color="#1f4e79")
ax.set_xlabel("Comments assigned")
ax.set_ylabel("Topic id")
ax.set_title("Selected topic model: comments per topic")
fig.tight_layout()
fig.savefig(FIG / "selected_topic_sizes.png", dpi=150)
plt.close(fig)

tasks = list(validation["task"].unique())
models = list(validation["model"].unique())
fig, ax = plt.subplots(figsize=(11, 5))
width = 0.15
for i, model in enumerate(models):
    part = validation[validation["model"] == model].set_index("task").loc[tasks]
    positions = [p + (i - len(models) / 2) * width for p in range(len(tasks))]
    ax.bar(positions, part["macro_f1"], width=width, label=model)
ax.set_xticks(range(len(tasks)))
ax.set_xticklabels(tasks, rotation=15)
ax.set_ylabel("Validation macro-F1")
ax.set_ylim(0, 1)
ax.set_title("Validation macro-F1 used to select each task model")
ax.legend(fontsize=8, ncol=3)
fig.tight_layout()
fig.savefig(FIG / "validation_macro_f1.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(heldout["task"], heldout["test_macro_f1"], color="#1f4e79", label="Held-out test")
ax.plot(agreement["task"], agreement["refit_agreement_macro_f1"], color="#b85c38", marker="o", label="Refit agreement")
ax.set_ylim(0, 1)
ax.set_ylabel("Macro-F1")
ax.set_title("Held-out test score and full-data refit agreement")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "heldout_and_refit_macro_f1.png", dpi=150)
plt.close(fig)

scopes = [
    "manual_annotated",
    "refit_prediction_on_annotated",
    "refit_prediction_on_all_comments",
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, task in zip(axes.ravel(), tasks):
    part = distribution[(distribution["task"] == task) & (distribution["scope"].isin(scopes))]
    labels = sorted(part["label"].unique())
    width = 0.25
    for i, scope in enumerate(scopes):
        scope_part = part[part["scope"] == scope].set_index("label")
        values = [float(scope_part["share"].get(label, 0)) for label in labels]
        positions = [p + (i - 1) * width for p in range(len(labels))]
        ax.bar(positions, values, width=width, label=scope if task == tasks[0] else None)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(task)
fig.suptitle("Label shares. Manual denominator is 1,199. Full-corpus denominator is 5,292.")
handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="upper center", ncol=3, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIG / "label_share_comparison.png", dpi=150)
plt.close(fig)

order = timeline.sort_values("publication_date")
fig, ax = plt.subplots(figsize=(10, 6))
colors = {
    "thematically related": "#1f4e79",
    "temporally associated": "#b85c38",
    "unlinked/unresolved": "#8aa0b4",
    "directly linked": "#2e7d32",
}
ax.barh(order["post_id"], order["n_comments"], color=[colors[value] for value in order["linkage_class"]])
ax.set_xlabel("Collected comments")
ax.set_title("Comments by source post, ordered by publication date")
fig.tight_layout()
fig.savefig(FIG / "post_timeline_volume.png", dpi=150)
plt.close(fig)

index = pd.DataFrame([
    ["topic_coherence.png", "c_npmi for every topic-model candidate on 5,292 comments"],
    ["selected_topic_sizes.png", "Comment counts for the selected topic model"],
    ["validation_macro_f1.png", "Validation macro-F1 for five models on four tasks"],
    ["heldout_and_refit_macro_f1.png", "Held-out test macro-F1 beside refit agreement"],
    ["label_share_comparison.png", "Manual and predicted label shares, with denominators in the title"],
    ["post_timeline_volume.png", "Comment counts by post. Color is the linkage class, not a causal result"],
], columns=["file", "reads_as"])
index.to_csv(OUT / "figure_index.csv", index=False)
print(index.to_string(index=False))


                          file                                                                reads_as
           topic_coherence.png                c_npmi for every topic-model candidate on 5,292 comments
      selected_topic_sizes.png                             Comment counts for the selected topic model
       validation_macro_f1.png                       Validation macro-F1 for five models on four tasks
heldout_and_refit_macro_f1.png                           Held-out test macro-F1 beside refit agreement
    label_share_comparison.png       Manual and predicted label shares, with denominators in the title
      post_timeline_volume.png Comment counts by post. Color is the linkage class, not a causal result


### Interpretation

Six figures are written. Topic coherence compares NMF, LDA, and BERTopic. The topic-size figure shows the six selected LDA topics. The validation figure shows why each task model was chosen. The held-out figure places test macro-F1 beside the higher refit agreement so the two numbers are not confused. The label-share figure uses 1,199 for the manual bars and 5,292 for the full-corpus bars. The timeline colors show linkage class, not a causal result.


The timeline colors encode the linkage class from notebook 07. No post is colored as directly linked. Comment counts are collected comments, not audience size.
